# House Prices — Регрессия цен на дома

**Соревнование:** [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)  
**Тема:** Регрессия  
**Инструменты:** `pandas`, `XGBRegressor`, target encoding

Предсказываем итоговую цену продажи домов в городе Эймс (Айова). В обучающей выборке 1460 домов и 79 признаков, описывающих почти всё: площадь, качество, район, возраст, гараж, подвал и т.д.

---
## Шаги
1. Загрузка данных
2. Обработка пропусков
3. Кодирование категориальных признаков
4. Генерация признаков (композитные признаки)
5. Target encoding для района
6. Обучение модели XGBoost
7. Оценка метрик
8. Генерация сабмита

## 1. Загрузка данных

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OrdinalEncoder

train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print('Размер train:', train.shape)
print('Размер test: ', test.shape)
train[['SalePrice', 'OverallQual', 'GrLivArea', 'Neighborhood']].head()

## 2. Обработка пропусков

Ключевые колонки с пропусками (в основном в тесте) заполняем нулём (числовые) или модой (категориальные).

In [ ]:
# Категориальные — заполняем модой из train
most_frequent_zone = train['MSZoning'].mode()[0]
train['MSZoning'] = train['MSZoning'].fillna(most_frequent_zone)
test['MSZoning']  = test['MSZoning'].fillna(most_frequent_zone)

# Числовые — заполняем нулём (нет гаража/подвала = 0 площади)
num_cols = ['TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
            'FullBath', 'HalfBath', 'GarageCars', 'GarageArea']

for col in num_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

print('Осталось пропусков в train:', train[num_cols].isnull().sum().sum())
print('Осталось пропусков в test: ', test[num_cols].isnull().sum().sum())

## 3. Кодирование категориальных признаков

`OrdinalEncoder` переводит текстовые категории в числа. Параметр `handle_unknown='use_encoded_value'` защищает от ошибок, если в тесте встретится категория, которой не было в обучении.

In [ ]:
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    dtype=int
)

for col in ['MSZoning', 'MSSubClass', 'LandContour', 'HouseStyle']:
    encoded_col = col + '_encoded'
    train[encoded_col] = encoder.fit_transform(train[[col]])
    test[encoded_col]  = encoder.transform(test[[col]])

print('Кодирование завершено.')

## 4. Генерация признаков

Вместо сырых признаков создаём композитные, которые объединяют несколько сигналов.

| Признак | Формула | Зачем |
|---|---|---|
| `FullArea` | GrLivArea + TotalBsmtSF | Общая жилая площадь сильнее, чем каждая по отдельности |
| `QualityScore` | OverallQual × OverallCond | Качество И состояние вместе |
| `BathroomScore` | сумма всех санузлов | Больше санузлов → выше цена |
| `GarageScore` | GarageCars × GarageArea | Вместимость гаража вместе с площадью |
| `House` | MSSubClass × LotArea × HouseStyle | Объединённый сигнал типа дома |

In [ ]:
for df in [train, test]:
    df['FullArea']      = df['GrLivArea'] + df['TotalBsmtSF']
    df['QualityScore']  = df['OverallQual'] * df['OverallCond']
    df['BathroomScore'] = df['BsmtFullBath'] + df['BsmtHalfBath'] + df['FullBath'] + df['HalfBath']
    df['GarageScore']   = df['GarageCars'] * df['GarageArea']
    df['House']         = df['MSSubClass'] * df['LotArea'] * df['HouseStyle_encoded']

print('Признаки сгенерированы.')
train[['FullArea', 'QualityScore', 'BathroomScore', 'GarageScore']].describe()

## 5. Target encoding для района

Район как сырая категория ничего не говорит модели об уровне цен. Target encoding заменяет каждый район на медианную цену продажи в нём — гораздо более информативный сигнал.

In [ ]:
neighborhood_price = train.groupby('Neighborhood')['SalePrice'].median()

train['NeighborhoodPrice'] = train['Neighborhood'].map(neighborhood_price)
test['NeighborhoodPrice']  = test['Neighborhood'].map(neighborhood_price)

# Заглушка для районов, которых не было в train
global_median = train['SalePrice'].median()
test['NeighborhoodPrice'] = test['NeighborhoodPrice'].fillna(global_median)

print('Топ-5 самых дорогих районов:')
print(neighborhood_price.sort_values(ascending=False).head())

## 6. Обучение модели XGBoost

**Почему XGBoost?**
- Автоматически ловит нелинейные зависимости
- Находит взаимодействия признаков (например, большая площадь + высокое качество = сильно выше цена)
- Устойчив к выбросам
- Стабильно выигрывает соревнования по регрессии

**Ключевые параметры против переобучения:**
- `max_depth=4` — неглубокие деревья лучше обобщают
- `subsample=0.8` — каждое дерево видит 80% данных (добавляет случайности)
- `colsample_bytree=0.8` — каждое дерево видит 80% признаков
- `min_child_weight=5` — для разбиения нужно минимум 5 объектов

In [ ]:
features = [
    'House', 'QualityScore', 'FullArea', 'YearBuilt',
    'BathroomScore', 'GarageScore', 'NeighborhoodPrice',
    'YearRemodAdd', 'LandContour_encoded'
]

X      = train[features]
y      = train['SalePrice']
X_test = test[features]

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42
)

model.fit(X, y)
print('Модель обучена.')

## 7. Оценка метрик

In [ ]:
# CV MAE — честная оценка (модель не видела эти данные при обучении)
mae_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
cv_mae = -mae_scores.mean()

# Метрики на train
y_pred = model.predict(X)
r2   = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print('=' * 50)
print(' МЕТРИКИ НА TRAIN:')
print(f'  R²   : {r2:.4f}')
print(f'  RMSE : ${rmse:,.2f}')
print('-' * 50)
print(' КРОСС-ВАЛИДАЦИЯ (5 фолдов):')
print(f'  MAE по фолдам: {[-round(s, 2) for s in mae_scores]}')
print(f'  Средний MAE : ${cv_mae:,.2f}')
print('=' * 50)

# Важность признаков
importances = sorted(zip(features, model.feature_importances_), key=lambda x: x[1], reverse=True)
print('\nВажность признаков:')
for name, imp in importances:
    print(f'  {name:<22}: {imp:.2%}')

## 8. Генерация сабмита

In [ ]:
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id':        test['Id'],
    'SalePrice': predictions
})

submission.to_csv('submission.csv', index=False)
print('submission.csv сохранён!')
print(f'Диапазон предсказанных цен: ${predictions.min():,.0f} — ${predictions.max():,.0f}')
submission.head()